# nuvT Reconstruction — Tres variantes de corrección ToF



In [1]:
import os, glob, shutil, time, json, concurrent.futures
import numpy as np
import uproot
import awkward as ak
import pandas as pd
import tensorflow as tf
import keras
from keras.layers import Dense, LayerNormalization, MultiHeadAttention, Dropout
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import norm as scipy_norm
import matplotlib.pyplot as plt
from IPython.display import display as ipy_display

try:
    from numba import jit, prange
    HAS_NUMBA = True
except ImportError:
    HAS_NUMBA = False
    def jit(*args, **kwargs):
        def decorator(func): return func
        return decorator
    prange = range

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
print('TF:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

2026-03-18 10:32:00.526431: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-18 10:32:00.526528: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-18 10:32:00.527642: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-18 10:32:00.532609: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF: 2.16.2
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2026-03-18 10:32:06.616689: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-18 10:32:06.653732: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-18 10:32:06.654066: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({
    'axes.labelsize':  14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'lines.linewidth': 2.0,
    'figure.dpi':      150,
})
MC_LABEL = 'MC Fall Production 2025'

### 1. Loading DATA from PosRecoCVNDataPrep_module.cc

In [2]:
C_LIGHT_CM_NS = 29.9792458  # cm/ns — ToF correction: nuvZ/c

# ─────────────────────────────────────────────────────────────────────────────
# Load data from ROOT (training_data_def_v1603.root)
# Split 80 / 10 / 10  (train / val / test)
# ─────────────────────────────────────────────────────────────────────────────

ROOT_FILE = '/exp/sbnd/data/users/svidales/AI_nuvT_project_support/AA_FinalPlots/training_data_def_v1603.root'
SEED      = 42

import time as _t
_t0 = _t.time()
print('Loading ROOT file...')
f_root = uproot.open(ROOT_FILE)

# ── 1. Channel geometry ───────────────────────────────────────────────────────
pds = f_root['dataprep/PDSMapTree'].arrays(
    ['OpDetID','OpDetType','OpDetX','OpDetY','OpDetZ'], entry_stop=312, library='np')
channel_geom_ref = {
    int(pds['OpDetID'][i]): (int(pds['OpDetType'][i]),
                              float(pds['OpDetX'][i]),
                              float(pds['OpDetY'][i]),
                              float(pds['OpDetZ'][i]))
    for i in range(312)}
print(f'  channel_geom_ref: {len(channel_geom_ref)} channels  ({_t.time()-_t0:.1f}s)')

# ── 2. Labels ─────────────────────────────────────────────────────────────────
AV = {'x': (-200.0, 200.0), 'y': (-200.0, 200.0), 'z': (0.0, 500.0)}

lab = f_root['dataprep/labels/tree'].arrays(
    ['passed_filters', 'selected_tpc',
     'nuv_t', 'nuv_x', 'nuv_y', 'nuv_z',
     'dEprom_x', 'dEprom_y', 'dEprom_z'], library='ak')

ok   = np.array(lab['passed_filters'], dtype=bool)
stpc = np.array(lab['selected_tpc'],   dtype=np.int32)
print(f'  passed_filters: {ok.sum():,} / {len(ok):,}  ({_t.time()-_t0:.1f}s)')

# nuvT/X/Y/Z: first neutrino inside the active volume (same criterion as C++ module)
in_av = ((lab['nuv_x'] >= AV['x'][0]) & (lab['nuv_x'] <= AV['x'][1]) &
          (lab['nuv_y'] >= AV['y'][0]) & (lab['nuv_y'] <= AV['y'][1]) &
          (lab['nuv_z'] >= AV['z'][0]) & (lab['nuv_z'] <= AV['z'][1]))

nuvT_all = np.array(ak.firsts(lab['nuv_t'][in_av])[ok], dtype=np.float32)
nuvX_all = np.array(ak.firsts(lab['nuv_x'][in_av])[ok], dtype=np.float32)
nuvY_all = np.array(ak.firsts(lab['nuv_y'][in_av])[ok], dtype=np.float32)
nuvZ_all = np.array(ak.firsts(lab['nuv_z'][in_av])[ok], dtype=np.float32)

# dEprom: indexed by TPC (always 2 elements: [TPC0, TPC1])
def get_tpc_label(arr_ak, ok_mask, tpc_idx):
    ok_arr = ak.to_list(arr_ak[ok_mask])
    return np.array([row[t] for row, t in zip(ok_arr, tpc_idx)], dtype=np.float32)

stpc_ok     = stpc[ok]
dEpromx_all = get_tpc_label(lab['dEprom_x'], ok, stpc_ok)
dEpromy_all = get_tpc_label(lab['dEprom_y'], ok, stpc_ok)
dEpromz_all = get_tpc_label(lab['dEprom_z'], ok, stpc_ok)
print(f'  labels extracted  ({_t.time()-_t0:.1f}s)')

# ── 3. Temporal features (N, 20, 7) ──────────────────────────────────────────
# Feature layout: [t_us, logPE_norm, det_type, x_norm, y_norm, z_norm, delta_t_us]
# t_us already ToF-corrected: t_ophit - nuvZ/c  (applied in C++ module)
temp_ak = f_root['dataprep/temporal/tree'].arrays(
    ['temporal_features', 'temporal_seq_len'], library='ak')
seq_len = int(ak.to_numpy(temp_ak['temporal_seq_len'][ok])[0])  # = 20
X_flat  = ak.to_numpy(temp_ak['temporal_features'][ok])         # (N_ok, 140)
X_all   = X_flat.reshape(-1, seq_len, 7).astype(np.float32)
print(f'  temporal features: {X_all.shape}  ({_t.time()-_t0:.1f}s)')

# ── 4. Label: nuvT_tof = nuvT - nuvZ/c [ns] ──────────────────────────────────
label_all = (nuvT_all - nuvZ_all / C_LIGHT_CM_NS).astype(np.float32)

# ── 4.5. Remove events with unphysical ophit times (t_us < 0) ────────────────
# t_us is ToF-corrected arrival time; negative values are unphysical.
# Check only real (non-padded) steps: padding uses -1000.0
_PADVAL   = -1000.0
_real     = X_all[:, :, 0] != _PADVAL                            # (N, 20) bool
_t_min    = np.where(_real, X_all[:, :, 0], np.inf).min(axis=1) # min t_us per event
phys_ok   = _t_min >= 0.0
n_removed = int((~phys_ok).sum())
print(f'  Events with t_us < 0: {n_removed:,} removed  ({phys_ok.sum():,} kept)  ({_t.time()-_t0:.1f}s)')

X_all       = X_all[phys_ok]
label_all   = label_all[phys_ok]
nuvT_all    = nuvT_all[phys_ok];   nuvX_all = nuvX_all[phys_ok]
nuvY_all    = nuvY_all[phys_ok];   nuvZ_all = nuvZ_all[phys_ok]
dEpromx_all = dEpromx_all[phys_ok]
dEpromy_all = dEpromy_all[phys_ok]
dEpromz_all = dEpromz_all[phys_ok]

# ── 5. Split 80 / 10 / 10 ────────────────────────────────────────────────────
N = len(label_all)
idx_tr, idx_tmp = train_test_split(np.arange(N), test_size=0.20, random_state=SEED)
idx_val, idx_te = train_test_split(idx_tmp,       test_size=0.50, random_state=SEED)
idx_tr.sort(); idx_val.sort(); idx_te.sort()

X_tof_tr  = X_all[idx_tr];  X_tof_val  = X_all[idx_val];  X_tof_te  = X_all[idx_te]

# Scale labels with MinMaxScaler — fit on train only, apply to val/test via inverse_transform
sc_tof       = MinMaxScaler()
y_tof_tr     = sc_tof.fit_transform(label_all[idx_tr] .reshape(-1,1)).flatten()
y_tof_val_sc = sc_tof.transform    (label_all[idx_val].reshape(-1,1)).flatten()
y_tof_val_ns = label_all[idx_val]   # unscaled, for report()
label_tof_te = label_all[idx_te]    # unscaled, for eval_on_test()

# Auxiliary arrays for analysis
def sp3(arr): return arr[idx_tr], arr[idx_val], arr[idx_te]
nuvT_training, nuvT_val, nuvT_test       = sp3(nuvT_all)
nuvX_training, nuvX_val, nuvX_test       = sp3(nuvX_all)
nuvY_training, nuvY_val, nuvY_test       = sp3(nuvY_all)
nuvZ_training, nuvZ_val, nuvZ_test       = sp3(nuvZ_all)
dEpromx_training, dEpromx_val, dEpromx_test = sp3(dEpromx_all)
dEpromy_training, dEpromy_val, dEpromy_test = sp3(dEpromy_all)
dEpromz_training, dEpromz_val, dEpromz_test = sp3(dEpromz_all)

print(f'\nTrain: {len(idx_tr):,}  Val: {len(idx_val):,}  Test: {len(idx_te):,}')
print(f'X_tof shape:   {X_tof_tr.shape}')
print(f'label range:   [{label_all.min():.0f}, {label_all.max():.0f}] ns')
print(f'Total time:    {_t.time()-_t0:.1f}s')

Loading ROOT file...
  channel_geom_ref: 312 channels  (0.0s)


  passed_filters: 554,528 / 1,365,732  (7.5s)
  labels extracted  (10.4s)
  temporal features: (554528, 20, 7)  (16.1s)
  Events with t_us < 0: 826 removed  (553,702 kept)  (16.2s)

Train: 442,961  Val: 55,370  Test: 55,371
X_tof shape:   (442961, 20, 7)
label range:   [55, 1580] ns
Total time:    16.4s


### 2. Normalization

In [3]:
from sklearn.preprocessing import MinMaxScaler as MinMaxScaler2

PAD_VALUE = -1000.0

# ── Normalize input features (fit on train only) ──────────────────────────────
# Features 3,4,5 (x/y/z) already normalized in C++; det_type (2) is categorical.
#   feat 0 — t_us     : MinMaxScaler [0,1]
#   feat 1 — PE       : log1p → MinMaxScaler [0,1]
#   feat 6 — delta_t  : recalculated from normalized t_us (same units, no extra scaler)

assert X_tof_tr[:, :, 0][X_tof_tr[:, :, 0] != PAD_VALUE].mean() > 0.01, \
    "t_us looks already normalized — reload data before running this cell"

def _real(X):
    """Boolean mask of non-padded slots: shape (N, seq_len)."""
    return X[:, :, 0] != PAD_VALUE

# ── feat 1: log1p(PE) in-place ────────────────────────────────────────────────
for X in [X_tof_tr, X_tof_val, X_tof_te]:
    m = _real(X)
    X[:, :, 1] = np.where(m, np.log1p(np.maximum(X[:, :, 1], 0.0)), PAD_VALUE)

# ── feat 0 + 1: MinMaxScaler [0,1] ───────────────────────────────────────────
sc_t  = MinMaxScaler2()
sc_pe = MinMaxScaler2()
sc_t .fit(X_tof_tr[:, :, 0][_real(X_tof_tr)].reshape(-1, 1))
sc_pe.fit(X_tof_tr[:, :, 1][_real(X_tof_tr)].reshape(-1, 1))

for sc, fi in [(sc_t, 0), (sc_pe, 1)]:
    for X in [X_tof_tr, X_tof_val, X_tof_te]:
        m = _real(X)
        vals = X[:, :, fi].copy()
        vals[m] = sc.transform(vals[m].reshape(-1, 1)).flatten()
        X[:, :, fi] = np.where(m, vals, PAD_VALUE)

# ── feat 6: delta_t recalculated from normalized t_us ────────────────────────
for X in [X_tof_tr, X_tof_val, X_tof_te]:
    m       = _real(X)
    t_norm  = X[:, :, 0]
    t_first = np.where(m, t_norm, np.inf).min(axis=1, keepdims=True)
    t_first = np.where(np.isinf(t_first), 0.0, t_first)
    X[:, :, 6] = np.where(m, t_norm - t_first, PAD_VALUE)

print(f'sc_t  : [{sc_t.data_min_[0]:.4f}, {sc_t.data_max_[0]:.4f}] µs')
print(f'sc_pe : [{sc_pe.data_min_[0]:.4f}, {sc_pe.data_max_[0]:.4f}] log(1+PE)')
print(f't_us    after norm — train min/max: {X_tof_tr[:,:,0][_real(X_tof_tr)].min():.3f} / {X_tof_tr[:,:,0][_real(X_tof_tr)].max():.3f}')
print(f'logPE   after norm — train min/max: {X_tof_tr[:,:,1][_real(X_tof_tr)].min():.3f} / {X_tof_tr[:,:,1][_real(X_tof_tr)].max():.3f}')
print(f'delta_t after norm — train min/max: {X_tof_tr[:,:,6][_real(X_tof_tr)].min():.3f} / {X_tof_tr[:,:,6][_real(X_tof_tr)].max():.3f}')

sc_t  : [0.0639, 7.1121] µs
sc_pe : [0.0000, 12.2186] log(1+PE)
t_us    after norm — train min/max: 0.000 / 1.000
logPE   after norm — train min/max: 0.000 / 1.000
delta_t after norm — train min/max: 0.000 / 0.948


## 4. Modelo

**transformer**

In [ ]:
class TBlock(keras.layers.Layer):
    def __init__(self, d, h, ff, dr, **kw):
        super().__init__(**kw)
        self.mha = MultiHeadAttention(num_heads=h, key_dim=d//h, dropout=dr)
        self.ff1 = Dense(ff, activation='gelu'); self.ff2 = Dense(d)
        self.ln1 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.drop = Dropout(dr)
        self._d=d; self._h=h; self._ff=ff; self._dr=dr
    def call(self, x, pm, am, training=False):
        x = self.ln1(x + self.mha(x, x, attention_mask=am, training=training))
        x = self.ln2(x + self.ff2(self.drop(self.ff1(x), training=training)))
        return x * pm[:,:,tf.newaxis]
    def get_config(self):
        c = super().get_config(); c.update({'d':self._d,'h':self._h,'ff':self._ff,'dr':self._dr}); return c


class FastTransformer(keras.Model):
    def __init__(self, d=64, h=4, ff=128, nb=2, dr=0.1, **kw):
        super().__init__(**kw)
        self.proj   = Dense(d, name='proj')
        self.blocks = [TBlock(d, h, ff, dr, name=f'b{i}') for i in range(nb)]
        self.head   = Dense(64, activation='relu')
        self.out    = Dense(1)
        self._cfg   = dict(d=d, h=h, ff=ff, nb=nb, dr=dr)
    def call(self, x, training=False):
        pm  = tf.cast(tf.not_equal(x[:,:,0], PAD_VALUE), tf.float32)
        am  = tf.cast(pm[:,tf.newaxis,:], tf.bool)
        out = self.proj(x) * pm[:,:,tf.newaxis]
        for b in self.blocks: out = b(out, pm, am, training=training)
        nr  = tf.maximum(tf.reduce_sum(pm, axis=1, keepdims=True), 1.0)
        pos = tf.cast(tf.range(tf.shape(out)[1]), tf.float32)[tf.newaxis,:]
        w   = tf.exp(-3.0*pos/nr) * pm
        w   = w / tf.maximum(tf.reduce_sum(w, axis=1, keepdims=True), 1e-6)
        out = tf.reduce_sum(out * w[:,:,tf.newaxis], axis=1)
        return self.out(self.head(out))
    def get_config(self):
        c = super().get_config(); c.update(self._cfg); return c


def make_transformer_model():
    m = FastTransformer(d=128, h=8, ff=256, nb=4, dr=0.1)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss=keras.losses.Huber(delta=0.05), metrics=['mae'])
    return m

_m = make_transformer_model(); _m(X_tof_tr[:2]); _m.summary()

**lstm**

In [ ]:
class ExpWeightedPool(keras.layers.Layer):
    """Pooling exponencial ponderado: da más peso a los ophits más tempranos."""
    def call(self, inputs):
        lstm_out, inp = inputs
        pm     = tf.cast(tf.not_equal(inp[:, :, 0], PAD_VALUE), tf.float32)
        n_real = tf.maximum(tf.reduce_sum(pm, axis=1, keepdims=True), 1.0)
        pos    = tf.cast(tf.range(tf.shape(lstm_out)[1]), tf.float32)[tf.newaxis, :]
        w      = tf.exp(-3.0 * pos / n_real) * pm
        w      = w / tf.maximum(tf.reduce_sum(w, axis=1, keepdims=True), 1e-6)
        return tf.reduce_sum(lstm_out * w[:, :, tf.newaxis], axis=1)

    def get_config(self):
        return super().get_config()


def make_lstm_model():
    seq_len, n_feat = X_tof_tr.shape[1], X_tof_tr.shape[2]
    inp      = keras.Input(shape=(seq_len, n_feat))
    x        = keras.layers.Masking(mask_value=PAD_VALUE)(inp)
    x        = keras.layers.LSTM(256, return_sequences=True, use_cudnn=False)(x)
    lstm_out = keras.layers.LSTM(128, return_sequences=True, use_cudnn=False)(x)
    pooled   = ExpWeightedPool()([lstm_out, inp])
    x        = Dense(64, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(0.01))(pooled)
    out      = keras.layers.Flatten()(Dense(1)(x))
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss=keras.losses.Huber(delta=0.05), metrics=['mae'])
    return m

_m2 = make_lstm_model(); _m2(X_tof_tr[:2]); _m2.summary()

## 5. Entrenar

In [6]:
def train_fast(model, Xtr, ytr, Xval, yval, epochs=200, batch=64, patience=20):
    cbs = [
        EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6, verbose=1),
        ModelCheckpoint('/tmp/nuvT_best.keras', monitor='val_loss', save_best_only=True, verbose=0),
    ]
    print(f'Train {len(Xtr):,}  Val {len(Xval):,}  batch={batch}')
    hist = model.fit(Xtr, ytr, validation_data=(Xval, yval),
                     epochs=epochs, batch_size=batch, callbacks=cbs, verbose=1)
    return model.predict(Xval, batch_size=batch, verbose=0).flatten(), hist


def report(pred_sc, y_ns, sc, name):
    pred_ns = sc.inverse_transform(pred_sc.reshape(-1,1)).flatten()
    diff    = pred_ns - y_ns
    mu, sig = scipy_norm.fit(np.clip(diff, -200, 200))
    print(f'  {name}:  bias={mu:.1f} ns   sigma={sig:.1f} ns   MAE={np.abs(diff).mean():.1f} ns')
    return diff, pred_ns

print('OK')

OK


In [7]:
print('='*55, '\nMODELO Transformer\n' + '='*55)
model_tof = make_transformer_model()
pred_tof_sc, hist_tof = train_fast(model_tof, X_tof_tr, y_tof_tr, X_tof_val, y_tof_val_sc)
diff_tof, pred_tof_ns = report(pred_tof_sc, y_tof_val_ns, sc_tof, 'ToF')

MODELO Transformer
Train 442,961  Val 55,370  batch=64
Epoch 1/200


2026-03-18 10:32:35.978941: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-18 10:32:36.738607: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


6922/6922 ━━━━━━━━━━━━━━━━━━━━ 64s 6ms/step - loss: 0.0095 - mae: 0.0491 - val_loss: 0.0058 - val_mae: 0.0476 - learning_rate: 0.0010
Epoch 2/200
6922/6922 ━━━━━━━━━━━━━━━━━━━━ 30s 4ms/step - loss: 0.0048 - mae: 0.0342 - val_loss: 0.0043 - val_mae: 0.0255 - learning_rate: 0.0010
Epoch 3/200
6922/6922 ━━━━━━━━━━━━━━━━━━━━ 30s 4ms/step - loss: 0.0044 - mae: 0.0290 - val_loss: 0.0043 - val_mae: 0.0247 - learning_rate: 0.0010
Epoch 4/200
6922/6922 ━━━━━━━━━━━━━━━━━━━━ 30s 4ms/step - loss: 0.0042 - mae: 0.0263 - val_loss: 0.0047 - val_mae: 0.0353 - learning_rate: 0.0010
Epoch 5/200
6922/6922 ━━━━━━━━━━━━━━━━━━━━ 29s 4ms/step - loss: 0.0042 - mae: 0.0254 - val_loss: 0.0044 - val_mae: 0.0316 - learning_rate: 0.0010
Epoch 6/200
6922/6922 ━━━━━━━━━━━━━━━━━━━━ 30s 4ms/step - loss: 0.0041 - mae: 0.0245 - val_loss: 0.0043 - val_mae: 0.0278 - learning_rate: 0.0010
Epoch 7/200
6922/6922 ━━━━━━━━━━━━━━━━━━━━ 30s 4ms/step - loss: 0.0041 - mae: 0.0238 - val_loss: 0.0040 - val_mae: 0.0212 - learning_rat

In [8]:
print('='*55, '\nMODELO LSTM\n' + '='*55)
model_lstm_tof = make_lstm_model()
pred_lstm_tof_sc, hist_lstm_tof = train_fast(model_lstm_tof, X_tof_tr, y_tof_tr, X_tof_val, y_tof_val_sc)
diff_lstm_tof, pred_lstm_tof_ns = report(pred_lstm_tof_sc, y_tof_val_ns, sc_tof, 'LSTM ToF')

MODELO LSTM
Train 442,961  Val 55,370  batch=64


/opt/conda/lib/python3.10/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'exp_weighted_pool_1' (of type ExpWeightedPool) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Epoch 1/200


6922/6922 ━━━━━━━━━━━━━━━━━━━━ 53s 7ms/step - loss: 0.0141 - mae: 0.0399 - val_loss: 0.0060 - val_mae: 0.0415 - learning_rate: 0.0010
Epoch 2/200
6922/6922 ━━━━━━━━━━━━━━━━━━━━ 45s 6ms/step - loss: 0.0051 - mae: 0.0307 - val_loss: 0.0048 - val_mae: 0.0230 - learning_rate: 0.0010
Epoch 3/200
6922/6922 ━━━━━━━━━━━━━━━━━━━━ 44s 6ms/step - loss: 0.0048 - mae: 0.0282 - val_loss: 0.0048 - val_mae: 0.0272 - learning_rate: 0.0010
Epoch 4/200
6922/6922 ━━━━━━━━━━━━━━━━━━━━ 44s 6ms/step - loss: 0.0046 - mae: 0.0270 - val_loss: 0.0047 - val_mae: 0.0298 - learning_rate: 0.0010
Epoch 5/200
6922/6922 ━━━━━━━━━━━━━━━━━━━━ 43s 6ms/step - loss: 0.0045 - mae: 0.0262 - val_loss: 0.0044 - val_mae: 0.0236 - learning_rate: 0.0010
Epoch 6/200
6922/6922 ━━━━━━━━━━━━━━━━━━━━ 43s 6ms/step - loss: 0.0044 - mae: 0.0256 - val_loss: 0.0044 - val_mae: 0.0227 - learning_rate: 0.0010
Epoch 7/200
6922/6922 ━━━━━━━━━━━━━━━━━━━━ 44s 6ms/step - loss: 0.0043 - mae: 0.0250 - val_loss: 0.0043 - val_mae: 0.0256 - learning_rat

**guardar modelo**

Saved in saved_model for inference in C++/Python, and in Keras as well in case it is necessary to retrain or fine-tune the model.

In [9]:
import os, joblib

SAVE_DIR = '/exp/sbnd/data/users/svidales/AI_nuvT_project_support/AA_FinalPlots/temporal/last_models_v170326_mse'
os.makedirs(SAVE_DIR, exist_ok=True)

# NOTE: models saved below were trained with Huber loss (keras.losses.Huber())

# SavedModel format — for inference (TF/C++)
model_tof.export(os.path.join(SAVE_DIR, 'transformer_tof'))
model_lstm_tof.export(os.path.join(SAVE_DIR, 'lstm_tof'))

# Keras format — for reloading and retraining
model_tof.save(os.path.join(SAVE_DIR, 'transformer_tof.keras'))
model_lstm_tof.save(os.path.join(SAVE_DIR, 'lstm_tof.keras'))

# Scalers
joblib.dump(sc_tof, os.path.join(SAVE_DIR, 'sc_tof.pkl'))
joblib.dump(sc_t,   os.path.join(SAVE_DIR, 'sc_t.pkl'))
joblib.dump(sc_pe,  os.path.join(SAVE_DIR, 'sc_pe.pkl'))

print(f'sc_tof : [{sc_tof.data_min_[0]:.2f}, {sc_tof.data_max_[0]:.2f}] ns')
print(f'sc_t   : [{sc_t.data_min_[0]:.4f}, {sc_t.data_max_[0]:.4f}] µs')
print(f'sc_pe  : [{sc_pe.data_min_[0]:.4f}, {sc_pe.data_max_[0]:.4f}] log(1+PE)')

INFO:tensorflow:Assets written to: /exp/sbnd/data/users/svidales/AI_nuvT_project_support/AA_FinalPlots/temporal/last_models_v170326_mse/transformer_tof/assets


INFO:tensorflow:Assets written to: /exp/sbnd/data/users/svidales/AI_nuvT_project_support/AA_FinalPlots/temporal/last_models_v170326_mse/transformer_tof/assets


Saved artifact at '/exp/sbnd/data/users/svidales/AI_nuvT_project_support/AA_FinalPlots/temporal/last_models_v170326_mse/transformer_tof'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 20, 7), dtype=tf.float32, name=None)
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140401106680672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140401101020976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140402886585760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140401110975408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140403529405312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140401110979632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140401110974704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140401110978752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140401110982624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  

/opt/conda/lib/python3.10/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'exp_weighted_pool_1' (of type ExpWeightedPool) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


INFO:tensorflow:Assets written to: /exp/sbnd/data/users/svidales/AI_nuvT_project_support/AA_FinalPlots/temporal/last_models_v170326_mse/lstm_tof/assets


INFO:tensorflow:Assets written to: /exp/sbnd/data/users/svidales/AI_nuvT_project_support/AA_FinalPlots/temporal/last_models_v170326_mse/lstm_tof/assets


Saved artifact at '/exp/sbnd/data/users/svidales/AI_nuvT_project_support/AA_FinalPlots/temporal/last_models_v170326_mse/lstm_tof'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 20, 7), dtype=tf.float32, name='keras_tensor_20')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140400843335888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140401110829184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140401110828128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140401110839216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140401110840448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140401110828656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140401110842384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140401105573616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140401110838512: TensorSpec(shape=(), dtype=tf.resource, name=No

## 6. Comparación en test

In [44]:
MODEL_DIR = '/exp/sbnd/data/users/svidales/AI_nuvT_project_support/AA_FinalPlots/temporal/last_models_v160326'

model_tof = keras.models.load_model(
    os.path.join(MODEL_DIR, 'transformer_tof.keras'),
    custom_objects={'FastTransformer': FastTransformer, 'TBlock': TBlock}
)

print('Transformer loaded from:', os.path.join(MODEL_DIR, 'transformer_tof.keras'))
model_tof.summary()

Transformer loaded from: /exp/sbnd/data/users/svidales/AI_nuvT_project_support/AA_FinalPlots/temporal/last_models_v160326/transformer_tof.keras


Model: "fast_transformer_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ proj (Dense)                    │ (None, 20, 128)        │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b0 (TBlock)                     │ ?                      │       132,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b1 (TBlock)                     │ ?                      │       132,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b2 (TBlock)                     │ ?                      │       132,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b3 (TBlock)                     │ ?                      │       132,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_54 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_55 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,617,797 (6.17 MB)

 Trainable params: 539,265 (2.06 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,078,532 (4.11 MB)

**comparación lstm vs transformer**

In [9]:
from scipy.optimize import curve_fit

BIAS_RANGE = 50
NBINS      = 200

def gaussian(x, A, mu, sigma):
    return A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def fit_residuals(diff, res_range=BIAS_RANGE, nbins=NBINS, n_sigma=2.5, n_iter=3):
    """Iterative Gaussian fit on the core of the residual distribution.

    Standard HEP approach: after the first fit, refit within [mu ± n_sigma*sigma]
    to avoid non-Gaussian tails biasing the resolution estimate.
    """
    bins = np.linspace(-res_range, res_range, nbins + 1)
    cnt, edges = np.histogram(diff, bins=bins)
    cx   = (edges[:-1] + edges[1:]) / 2

    # Initial guess from histogram
    peak  = cnt.max()
    mu0   = cx[cnt.argmax()]
    above = cx[cnt >= peak / 2]
    sig0  = (above[-1] - above[0]) / 2.35 if len(above) > 1 else 5.0

    popt = [peak, mu0, sig0]
    perr = [0, 0, 0]

    for _ in range(n_iter):
        mu_c, sig_c = popt[1], abs(popt[2])
        lo = max(mu_c - n_sigma * sig_c, -res_range)
        hi = min(mu_c + n_sigma * sig_c,  res_range)
        mask = (cx >= lo) & (cx <= hi) & (cnt > 0)
        if mask.sum() < 5:
            break
        try:
            popt, pcov = curve_fit(gaussian, cx[mask], cnt[mask],
                                   p0=popt,
                                   bounds=([0, -res_range, 0.1],
                                           [np.inf, res_range, res_range]),
                                   maxfev=10000)
            perr = np.sqrt(np.diag(pcov)).tolist()
        except Exception as e:
            print(f'  ! Fit failed: {e}')
            break

    return popt, perr, bins, cnt, cx

def eval_on_test(model, X_te, label_te, sc, batch=512):
    pred_sc = model.predict(X_te, batch_size=batch, verbose=0).flatten()
    pred_ns = sc.inverse_transform(pred_sc.reshape(-1, 1)).flatten()
    return pred_ns - label_te, pred_ns

print('OK — iterative Gaussian fit (n_sigma=2.5, n_iter=3)')


OK — iterative Gaussian fit (n_sigma=2.5, n_iter=3)


In [10]:
FIG_DIR = '/exp/sbnd/data/users/svidales/AI_nuvT_project_support/AA_FinalPlots/temporal/plots/mse'
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# ── Comparación LSTM vs Transformer — ToF (test set) ────────────────────────
print(">> Prediciendo en test set...")
diff_lstm_tof_te,  pred_lstm_ns  = eval_on_test(model_lstm_tof, X_tof_te, label_tof_te, sc_tof)
diff_tof_te,       pred_tof_ns   = eval_on_test(model_tof,      X_tof_te, label_tof_te, sc_tof)

truth_ns = label_tof_te

COLORS = {'LSTM': 'steelblue', 'Transformer': 'darkorange'}
LS_MAP  = {'LSTM': '-',        'Transformer': '--'}
models_test = [
    ('LSTM',        pred_lstm_ns,  hist_lstm_tof),
    ('Transformer', pred_tof_ns,   hist_tof),
]

print(f"\n{'='*75}")
print(f"  {'Model':<14} {'MAE':>8} {'RMSE':>9} {'Bias':>9} {'sigma_raw':>10} {'mu_gauss':>10} {'sigma_gauss':>12}")
print(f"  {'-'*14} {'-'*8} {'-'*9} {'-'*9} {'-'*10} {'-'*10} {'-'*12}")

fit_results = {}
for name, pred_ns, _ in models_test:
    res  = pred_ns - truth_ns
    mae  = np.abs(res).mean()
    rmse = np.sqrt(np.mean(res**2))
    popt, perr, bins, cnt, cx = fit_residuals(res)
    fit_results[name] = (popt, perr, bins, cnt, cx, res)
    print(f"  {name:<14} {mae:>8.2f} {rmse:>9.2f} {np.mean(res):>9.2f} "
          f"{np.std(res):>10.2f} {popt[1]:>10.2f} {abs(popt[2]):>12.2f}  ns")
print(f"{'='*75}")

# ── Figure 3x3 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(19, 16))

CMIN = 6
RMIN = float(truth_ns.min()) - 20
RMAX = float(truth_ns.max()) + 20
rng  = [[RMIN, RMAX]] * 2

# ── Row 0: Loss curves ────────────────────────────────────────────────────────
for col, (name, _, hist) in enumerate(models_test):
    ax = axes[0, col]
    ax.plot(hist.history['loss'],     label='train', color=COLORS[name], lw=2)
    ax.plot(hist.history['val_loss'], label='val',   color=COLORS[name], lw=2, ls='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Huber loss')
    ax.set_title(f'{name} \u2014 loss curves')
    ax.legend(); ax.grid(True, ls='--', alpha=0.4); ax.set_yscale('log')

ax = axes[0, 2]
for name, _, hist in models_test:
    ax.plot(hist.history['val_loss'], label=name, color=COLORS[name], lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Val Huber loss')
ax.set_title('Val loss \u2014 head-to-head')
ax.legend(); ax.grid(True, ls='--', alpha=0.4); ax.set_yscale('log')

# ── Row 1: 2D truth-vs-reco + metrics bar chart ───────────────────────────────
for col, (name, pred_ns, _) in enumerate(models_test):
    ax = axes[1, col]
    h  = ax.hist2d(truth_ns, pred_ns, bins=100, cmap='viridis', cmin=CMIN, range=rng)
    fig.colorbar(h[3], ax=ax, label='# Events')
    ax.plot([RMIN, RMAX], [RMIN, RMAX], 'r-', lw=2, alpha=0.8, label='Perfect prediction')
    ax.set_xlabel('Truth neutrino int. time (ToF corr.) [ns]')
    ax.set_ylabel('Reco neutrino int. time (ToF corr.) [ns]')
    ax.set_title(name, color=COLORS[name], fontweight='bold')
    ax.set_aspect('equal', adjustable='box'); ax.legend()

ax = axes[1, 2]
metric_labels = ['MAE', 'RMSE', 'sigma_raw', 'sigma_gauss', '|bias|']
vals = {}
for name, pred_ns, _ in models_test:
    res  = pred_ns - truth_ns
    popt = fit_results[name][0]
    vals[name] = [np.abs(res).mean(), np.sqrt(np.mean(res**2)),
                  np.std(res), abs(popt[2]), abs(np.mean(res))]
x = np.arange(len(metric_labels)); w = 0.35
for i, (name, _, __) in enumerate(models_test):
    bars = ax.bar(x + (i - 0.5) * w, vals[name], w,
                  color=COLORS[name], alpha=0.85, label=name)
    for b, v in zip(bars, vals[name]):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
                f'{v:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(metric_labels)
ax.set_ylabel('Value [ns]')
ax.set_title('Metrics summary (test)', fontweight='bold')
ax.legend(); ax.grid(True, ls='--', alpha=0.4, axis='y')

# ── Row 2: Gaussian fits + head-to-head normalized ───────────────────────────
for col, (name, pred_ns, _) in enumerate(models_test):
    ax = axes[2, col]
    popt, perr, bins, cnt, cx = fit_results[name][:5]
    ax.bar(cx, cnt, width=bins[1]-bins[0], color=COLORS[name], alpha=0.7, label='Residuals')
    x_fit = np.linspace(bins[0], bins[-1], 2000)
    ax.plot(x_fit, gaussian(x_fit, *popt), 'k-', lw=2.5, label='Gaussian fit')
    ax.axvline(0,       color='red',  lw=1.5, ls='--', alpha=0.7)
    ax.axvline(popt[1], color='navy', lw=1.5, ls=':',  alpha=0.9)
    ax.text(0.97, 0.95,
            f'\u03bc = {popt[1]:.2f} \u00b1 {perr[1]:.2f} ns\n\u03c3 = {abs(popt[2]):.2f} \u00b1 {perr[2]:.2f} ns',
            transform=ax.transAxes, fontsize=12, va='top', ha='right', family='monospace',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='gold', edgecolor='goldenrod', lw=2, alpha=0.92))
    ax.set_title(f'{name} \u2014 Gaussian fit (test)', color=COLORS[name], fontweight='bold')
    ax.set_xlabel('Residual reco \u2212 truth [ns]')
    ax.set_ylabel('# Events')
    ax.legend(); ax.grid(True, ls='--', alpha=0.4)

ax = axes[2, 2]
for name, pred_ns, _ in models_test:
    popt, perr, bins, cnt, cx = fit_results[name][:5]
    ax.step(cx, cnt / cnt.max(), where='mid', color=COLORS[name], lw=1.5, alpha=0.35)
    x_fit = np.linspace(bins[0], bins[-1], 2000)
    ax.plot(x_fit, gaussian(x_fit, *popt) / cnt.max(), color=COLORS[name], lw=2.5, ls=LS_MAP[name],
            label=f"{name}: \u03bc={popt[1]:.1f}, \u03c3={abs(popt[2]):.1f} ns")
ax.axvline(0, color='red', lw=1.5, ls='--', alpha=0.7)
ax.set_title('Head-to-head (normalized)')
ax.set_xlabel('Residual reco \u2212 truth [ns]')
ax.set_ylabel('Normalized events')
ax.legend(loc='upper right'); ax.grid(True, ls='--', alpha=0.4)

fig.text(0.01, 0.01, MC_LABEL, fontsize=10, style='italic', va='bottom')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'lstm_vs_transformer_comparison.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'lstm_vs_transformer_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
RF_PERIOD_NS = 18.936

zoom_min, zoom_max = 275, 490
hist_min, hist_max = 260, 500   # wider range to avoid edge cuts

t0 = zoom_min + (RF_PERIOD_NS - (zoom_min % RF_PERIOD_NS))
bunch_lines = np.arange(t0, zoom_max, RF_PERIOD_NS)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for col, (name, pred_ns) in enumerate([('LSTM', pred_lstm_ns), ('Transformer', pred_tof_ns)]):
    color = COLORS[name]
    ax    = axes[col]

    ax.hist(truth_ns, bins=300, range=(hist_min, hist_max),
            histtype='step', color='darkorange', lw=2.0, label='Truth', zorder=2)
    ax.hist(pred_ns, bins=300, range=(hist_min, hist_max),
            color=color, alpha=0.75, label='Predicted', zorder=3)

    for t_b in bunch_lines:
        ax.axvline(t_b, color='red', alpha=0.25, linewidth=0.9, linestyle='--', zorder=1)

    ax.set_xlim(zoom_min, zoom_max)
    ax.set_title(f'{name}  \u2014  Zoom [{zoom_min}, {zoom_max}] ns  (BNB spacing \u2248 18.9 ns)',
                 color=color, fontweight='bold')
    ax.set_xlabel('Neutrino interaction time (ToF corrected) [ns]')
    ax.set_ylabel('Events')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.set_facecolor('whitesmoke')

fig.text(0.01, 0.01, MC_LABEL, fontsize=10, style='italic', va='bottom')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'lstm_vs_transformer_bnb_bunches.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'lstm_vs_transformer_bnb_bunches.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import os
nuvT_tof_all = (nuvT_all - nuvZ_all / C_LIGHT_CM_NS)

nuvT_us     = nuvT_all     / 1e3
nuvT_tof_us = nuvT_tof_all / 1e3

# nanoseconds versions (for bunch structure plot)
nuvT_ns     = nuvT_all
nuvT_tof_ns = nuvT_tof_all

SPILL_MIN_US, SPILL_MAX_US = 0.0, 1.6
RF_PERIOD_US = 18.936e-3
RF_PERIOD_NS = 18.936

FIG_DIR_MAIN = '/exp/sbnd/data/users/svidales/AI_nuvT_project_support/AA_FinalPlots/temporal'
os.makedirs(FIG_DIR_MAIN, exist_ok=True)

# ── Figure 1: raw vs ToF corrected ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, data, title in zip(axes,
                            [nuvT_us, nuvT_tof_us],
                            ['MC neutrino interaction time',
                             'MC neutrino interaction time (ToF corrected)']):
    ax.axvspan(SPILL_MIN_US, SPILL_MAX_US, color='salmon', alpha=0.18,
               label=f'MC spill [{SPILL_MIN_US}, {SPILL_MAX_US}] \u00b5s')
    ax.hist(data, bins=100, density=True, color='steelblue',
            alpha=0.85, edgecolor='white', linewidth=0.4)
    ax.set_title(title)
    ax.set_xlabel(r'$t_{\nu,\,\mathrm{int}}$ (\u00b5s)')
    ax.set_ylabel('Density of events')
    ax.legend()
    ax.grid(True, ls='--', alpha=0.3)

fig.text(0.01, 0.01, MC_LABEL, fontsize=10, style='italic', va='bottom')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR_MAIN, 'nuvT_tof_correction.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR_MAIN, 'nuvT_tof_correction.png'), dpi=300, bbox_inches='tight')
plt.show()

# ── Figure 2: BNB bunch structure (nanoseconds) ───────────────────────────────
zoom_min_ns, zoom_max_ns = 275, 490
hist_min_ns, hist_max_ns = 260, 500

# Find bunch offset from data: peak of first bunch in ToF corrected histogram
cnt_ref, edges_ref = np.histogram(nuvT_tof_ns, bins=2000,
                                   range=(hist_min_ns, hist_min_ns + RF_PERIOD_NS * 2))
cx_ref = (edges_ref[:-1] + edges_ref[1:]) / 2
first_peak_ns = cx_ref[cnt_ref.argmax()]
bunch_lines_ns = np.arange(first_peak_ns, zoom_max_ns + RF_PERIOD_NS, RF_PERIOD_NS)
bunch_lines_ns = bunch_lines_ns[(bunch_lines_ns >= zoom_min_ns) & (bunch_lines_ns <= zoom_max_ns)]
print(f'Bunch offset: {first_peak_ns:.2f} ns  (first peak of ToF corrected)')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(nuvT_ns,     bins=300, range=(hist_min_ns, hist_max_ns),
        density=True, histtype='step', color='steelblue',  lw=2,
        label=r'Raw $t_{\nu,\,\mathrm{int}}$')
ax.hist(nuvT_tof_ns, bins=300, range=(hist_min_ns, hist_max_ns),
        density=True, histtype='step', color='darkorange', lw=2,
        label=r'ToF corrected $t_{\nu,\,\mathrm{int}}$')
for t_b in bunch_lines_ns:
    ax.axvline(t_b, color='red', alpha=0.4, lw=1.0, ls='--')
ax.set_xlim(zoom_min_ns, zoom_max_ns)
ax.set_title('BNB bunch structure', fontweight='bold')
ax.set_xlabel(r'$t_{\nu,\,\mathrm{int}}$ (ns)')
ax.set_ylabel('Density of events')
ax.legend()
ax.grid(True, ls='--', alpha=0.3)
fig.text(0.01, 0.01, MC_LABEL, fontsize=10, style='italic', va='bottom')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR_MAIN, 'nuvT_bnb_bunch_structure.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR_MAIN, 'nuvT_bnb_bunch_structure.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f'Mean ToF shift: {(nuvT_us - nuvT_tof_us).mean()*1e3:.2f} ns  (= mean nuvZ/c)')
print(f'Figures saved in {FIG_DIR_MAIN}')